# Ⅰ第5回 宿題：転移学習を図解する（知識の整理）

**問い**：転移学習で，何を・なぜ・どこまで学習し直すのか．自分で調べて整理し，Google スライドで図解する．

- 授業のスライドは例の 1 つ．**授業スライド以外の資料を 2 つ以上**使って調べる
- 提出物：この `summary.ipynb` と，Google スライドを PDF にした `c1_d5_summary.pdf`
- 期限：11/5（木）13:10．「第5回 課題提出」へ

## (1) 調べる

資料の例（ほかの資料でもよい）：

- 教科書 第8章「転移学習プログラミング」
- PyTorch 公式チュートリアル「Transfer Learning for Computer Vision Tutorial」：https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html
- スタンフォード大学の講義ノート CS231n「Transfer Learning」：https://cs231n.github.io/transfer-learning/
- 論文 Yosinski ら（2014）「How transferable are features in deep neural networks?」：https://arxiv.org/abs/1411.1792

LLM（ChatGPT など）で調べてもよい．ただし LLM の答えは出典にならない．書いた内容は上のような資料で確かめ，その資料を出典に書く．

## (2) 整理する（スライドを作る前に書く）

このセルをダブルクリックして，＿＿ に書き込む．1 マス 1〜2 行．

| テーマ | 自分の言葉の説明 | 根拠（資料名と URL・章） | スライドの何枚目 |
|---|---|---|---|
| 1. 転移学習とは何か（ゼロからの学習との違い） | 大量のデータ（ImageNet 約 128 万枚）で学習済みのモデルの重みを出発点にして，自分のタスクに合わせて学習し直すこと．ゼロからの学習は重みをランダムから始めるので大量のデータが要るが，転移学習は少ないデータでも高い正解率が出やすい | CS231n「Transfer Learning」https://cs231n.github.io/transfer-learning/ ／ 教科書 第8章 | 1  |
| 2. 凍結・ファインチューニング・フルファインチューニングの違い（資料による呼び方の違いも） | 凍結は学習済みの層を固定し，付け替えた最後の層だけ学習する．ファインチューニングは学習済みの重みを出発点にして，一部または全部の層を小さい学習率で学習し直す．全部の層を学習し直すことを特にフルファインチューニングと呼ぶ．PyTorch のチュートリアルでは凍結を「ConvNet as fixed feature extractor」，全部を学習し直すことを「Finetuning the ConvNet」と呼び，授業では「凍結」「微調整」と呼んでいる | PyTorch「Transfer Learning for Computer Vision Tutorial」https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html ／ CS231n | 2 |
| 3. 学習済みの重みが別のタスクで役立つ理由（入口の層と出口の層の役割） | 入口に近い層は，線の向きや色のかたまりのような，どの画像にも共通する一般的な特徴を取り出す．出口に近い層ほど，学習したタスク（ImageNet の 1000 種類）に特化した特徴になる．一般的な特徴はほかのタスクでもそのまま使えるので，学習済みの重みが役立つ | Yosinski ら（2014）「How transferable are features in deep neural networks?」https://arxiv.org/abs/1411.1792 | 3 |
| 4. 微調整の学習率を小さくする理由（破滅的忘却） | 学習済みの重みはすでに良い特徴を取り出せる状態なので，大きな学習率で全部の層を動かすと，その特徴が壊れて，前に学習した内容を忘れてしまう（破滅的忘却）．小さな学習率で少しずつ直せば，良い特徴を残したまま自分のタスクに合わせられる | CS231n「Transfer Learning」（Learning rates）／ 授業 演習1（微調整は lr=0.0001） | 4 |
| 5. 何層まで学習し直すかの決め方 | データの量と，ImageNet との似ている度合いの 2 つで決める．データが少なく似ているなら最後の層だけ（凍結），データが多く似ているなら全体を微調整，データが少なく似ていないなら手前の層の特徴を使って最後の層を学習，データが多く似ていないなら全体を微調整かゼロから学習する | CS231n「Transfer Learning」（When and how to fine-tune?） | 5 |
| 6. 自分の写真ならどうするか（(3) を根拠に） | 自分の靴の写真は 100 枚と少なく，ImageNet にも靴のクラスがあって似ているので，最後の層だけ学習する凍結か，出口に近い層だけを学習し直すのがよいと考えた | 自分の演習2 の結果 ／ CS231n | 6 |

## (3) 自分の写真ならどうするか

演習2（`ex2.ipynb`）の結果を写す．

| モデル | 平均 ± 標準偏差（%） | CNN との 差 ÷ 標準偏差 と判定 |
|---|---|---|
| CNN（ゼロから） | 61.3 ± 17.3 | ― |
| ResNet18 凍結 | 76.0 ± 6.0 | 14.7 ÷ 12.9 = 1.1 どちらとも言えない |
| ResNet18 微調整 | 85.3 ± 12.8 | 24.0 ÷ 15.2 = 1.6 どちらとも言えない |
| ViT 凍結 | 82.7 ± 14.6 | 21.3 ÷ 16.0 = 1.3 どちらとも言えない |

- 自分の写真は，「写真が多い・少ない」×「ImageNet に似ている・似ていない」のどれに当たるか．理由：「写真が少ない × ImageNet に似ている」に当たる．写真は 2 クラスで 100 枚しかなく，ImageNet にもランニングシューズ・ローファー・サンダルなど靴のクラスがあるので，学習済みの特徴がそのまま使えると考えた．
- ResNet18 なら，何層まで学習し直すのがよいと考えるか．理由：最後の層だけ（凍結）か，出口に近い最後のブロック（layer4）と最後の層までがよいと考える．CS231n では「少ない × 似ている」なら最後の層だけを学習するのがよいとされていて，自分の結果でも凍結は標準偏差が 6.0 と一番小さく安定していた．微調整は平均 85.3% と一番高かったが，100 枚で 1,117 万個の重みを全部動かすと過学習しやすく，シードによって 66.7%〜100% とぶれが大きかった．
- それを確かめるには，どう実験するか（学習する層・回数・何で判定するか）：凍結（fc だけ），layer4 と fc，全部の 3 つの条件で，シード 0〜4 の 5 回ずつ学習し，テスト正解率の平均と標準偏差を比べ，差 ÷ 標準偏差 が 2 以上かで判定する．テストが 15 枚だと 1 枚で 6.7 ポイント動くので，写真を増やしてテストの枚数も増やす．

## (4) Google スライドで図解する

1. Google スライドで **5〜7 枚** 作る．1 枚に 1 テーマ（(2) の 1〜6）と，最後に出典の 1 枚
2. 各スライドに **図を 1 つ以上** 入れる．自分で描いても，Google スライドの AI（Gemini）に描かせてもよい
3. AI に描かせた図は，層の数・矢印・文字の誤りを確かめて直す．直せない所は説明文で正しく補い，図の近くに「AI で生成」と書く
4. 説明文は自分の言葉で書く
5. 人の顔・個人が分かるものは載せない
6. 「ファイル → ダウンロード → PDF ドキュメント」で保存し，ファイル名を `c1_d5_summary.pdf` にする

## (5) 評価の観点（各 ○・△・×）

| 観点 | 見る所 |
|---|---|
| 1. 定義 | 用語の定義が正確で，資料による呼び方の違いに触れている |
| 2. 違い | ゼロから・凍結・微調整の違いを，出発点と学習する層で説明している |
| 3. 理由 | 学習率を小さくする理由・層の役割を，根拠つきで説明している |
| 4. 図 | 図に誤りが無い（AI で描いた図は誤りを直し，「AI で生成」と書いてある） |
| 5. 当てはめ | 自分の写真の結果（演習2）を使って，何層まで学習するかを考えている |
| 6. 出典 | 授業スライド以外の資料を 2 つ以上，名前と URL（または章）で示している |
| 7. 自分の言葉 | 説明文を自分の言葉で書いている（資料や AI の文をそのまま写していない） |

**提出前チェック**：(2) と (3) の ＿＿ をすべて埋めた／PDF は 5〜7 枚で，各スライドに図がある／授業スライド以外の出典が 2 つ以上ある